# News Trend Detection Pipeline

Notebook này xây dựng hệ thống phát hiện xu hướng tin tức từ nhiều nguồn báo điện tử Việt Nam.

Pipeline gồm các bước:

1. Thu thập RSS từ nhiều tờ báo.
2. Crawl nội dung bài viết.
3. Sinh embedding bằng BGE-M3.
4. Gom cụm chủ đề bằng UMAP + HDBSCAN.
5. Chọn các bài đại diện cho từng cụm.
6. Sử dụng LLM để sinh mô tả xu hướng.
7. Export kết quả dưới dạng JSON phục vụ dashboard hoặc API.

Kết quả đầu ra là danh sách các cụm chủ đề (trends), mỗi cụm bao gồm:

- Tên xu hướng
- Số lượng bài viết
- Các bài viết đại diện
- Danh sách toàn bộ bài báo thuộc cụm
- Các metric đánh giá chất lượng clustering

# Environment Setup

Cài đặt các thư viện cần thiết cho:

- RSS parsing
- Web scraping
- Embedding generation
- Dimensionality reduction
- Density-based clustering
- LLM inference
- Export dữ liệu

In [1]:
!pip install -q feedparser beautifulsoup4 tqdm sentence-transformers umap-learn hdbscan xlsxwriter bitsandbytes accelerate transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.3 MB/s eta 0:00:00


In [2]:
import requests
import feedparser
import pandas as pd
import numpy as np
import json
from collections import defaultdict
import matplotlib.pyplot as plt

from bs4 import BeautifulSoup
from tqdm import tqdm
import re

import time
from datetime import datetime
from dateutil import parser
from concurrent.futures import ThreadPoolExecutor, as_completed

import umap
import hdbscan

from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.metrics.pairwise import cosine_similarity

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

session = requests.Session()

2026-06-22 15:58:07.556570: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782143887.811644      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782143887.885346      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782143888.542933      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782143888.542989      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782143888.542992      23 computation_placer.cc:177] computation placer alr

# RSS Collection

Dữ liệu đầu vào được lấy từ RSS của nhiều cơ quan báo chí Việt Nam.

Các nguồn được chia thành các nhóm chủ đề:

- Thế giới
- Thời sự
- Kinh doanh
- Công nghệ
- Thể thao
- Giải trí
- Giáo dục
- Sức khỏe
- Du lịch
- Xe

Mỗi RSS feed chỉ lấy một số lượng bài viết giới hạn để tránh crawl quá nhiều dữ liệu trong một lần chạy.

In [3]:
TOPIC_RSS = {

    "world": [
        "https://vnexpress.net/rss/the-gioi.rss",
        "https://thanhnien.vn/rss/the-gioi.rss",
        "https://tuoitre.vn/rss/the-gioi.rss",
        "https://tienphong.vn/rss/the-gioi-5.rss",
        "https://nhandan.vn/rss/thegioi-1231.rss",
        "https://vtv.vn/rss/the-gioi.rss"
    ],

    "news": [
        "https://vnexpress.net/rss/thoi-su.rss",
        "https://thanhnien.vn/rss/thoi-su.rss",
        "https://tuoitre.vn/rss/thoi-su.rss",
    ],

    "sports": [
        "https://vnexpress.net/rss/the-thao.rss",
        "https://thanhnien.vn/rss/the-thao.rss",
        "https://tuoitre.vn/rss/the-thao.rss",
        "https://tienphong.vn/rss/the-thao-11.rss",
        "https://nhandan.vn/rss/thethao-1224.rss",
        "https://vtv.vn/rss/the-thao.rss"
    ],

    "technology": [
        "https://vnexpress.net/rss/khoa-hoc-cong-nghe.rss",
        "https://thanhnien.vn/rss/cong-nghe.rss",
        "https://tuoitre.vn/rss/cong-nghe.rss",
        "https://nhandan.vn/rss/khoahoc-congnghe-1292.rss",
        "https://vtv.vn/rss/cong-nghe.rss"
    ],

    "business": [
        "https://vnexpress.net/rss/kinh-doanh.rss",
        "https://thanhnien.vn/rss/kinh-te.rss",
        "https://tuoitre.vn/rss/kinh-doanh.rss",
        "https://tienphong.vn/rss/kinh-te-3.rss",
        "https://nhandan.vn/rss/kinhte-1185.rss",
        "https://vtv.vn/rss/kinh-te.rss"
    ],

    "entertainment": [
        "https://vnexpress.net/rss/giai-tri.rss",
        "https://thanhnien.vn/rss/giai-tri.rss",
        "https://tuoitre.vn/rss/giai-tri.rss",
        "https://tienphong.vn/rss/giai-tri-36.rss",
        "https://vtv.vn/rss/van-hoa-giai-tri.rss"
    ],

    "education": [
        "https://vnexpress.net/rss/giao-duc.rss",
        "https://thanhnien.vn/rss/giao-duc.rss",
        "https://tuoitre.vn/rss/giao-duc.rss",
        "https://tienphong.vn/rss/giao-duc-71.rss",
        "https://nhandan.vn/rss/giaoduc-1303.rss",
        "https://vtv.vn/rss/giao-duc.rss"
    ],

    "health": [
        "https://vnexpress.net/rss/suc-khoe.rss",
        "https://thanhnien.vn/rss/suc-khoe.rss",
        "https://tuoitre.vn/rss/suc-khoe.rss",
        "https://tienphong.vn/rss/suc-khoe-210.rss",
        "https://nhandan.vn/rss/y-te-1309.rss",
        "https://vtv.vn/rss/y-te.rss"
    ],

    "travelling": [
        "https://vnexpress.net/rss/du-lich.rss",
        "https://thanhnien.vn/rss/du-lich.rss",
        "https://tuoitre.vn/rss.htm",
        "https://tienphong.vn/rss/hang-khong-du-lich-220.rss",
        "https://nhandan.vn/rss/du-lich-1257.rss"
    ],

    "vehicles": [
        "https://vnexpress.net/rss/oto-xe-may.rss",
        "https://thanhnien.vn/rss/xe.rss",
        "https://tuoitre.vn/xe.rss",
        "https://tienphong.vn/rss/xe-113.rss"
    ]
}

## Thu thập RSS Metadata

Ở bước này chỉ lấy metadata cơ bản:

- Tiêu đề
- URL bài viết
- Thời gian xuất bản
- Chủ đề
- RSS nguồn

Chưa crawl nội dung chi tiết.

In [4]:
def collect_articles(topic_rss, limit_per_feed=20):

    articles = []

    for topic, rss_list in topic_rss.items():

        for rss_url in rss_list:

            try:

                feed = feedparser.parse(rss_url)

                for entry in feed.entries[:limit_per_feed]:

                    articles.append({

                        "topic": topic,
                        "rss_source": rss_url,
                        "title": entry.get("title", ""),
                        "url": entry.get("link", ""),
                        "published": entry.get("published", "")

                    })

            except Exception as e:

                print("RSS error:", rss_url)

    return pd.DataFrame(articles)

In [5]:
rss_df = collect_articles(
    TOPIC_RSS,
    limit_per_feed=50
)

rss_df.info()
rss_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2550 entries, 0 to 2549
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   topic       2550 non-null   object
 1   rss_source  2550 non-null   object
 2   title       2550 non-null   object
 3   url         2550 non-null   object
 4   published   2550 non-null   object
dtypes: object(5)
memory usage: 99.7+ KB


,topic,rss_source,title,url,published
0,world,https://vnexpress.net/rss/the-gioi.rss,Ông Vance: Mỹ - Iran đã tạo được nền móng tốt ...,https://vnexpress.net/ong-vance-my-iran-da-tao...,"Mon, 22 Jun 2026 21:49:31 +0700"
1,world,https://vnexpress.net/rss/the-gioi.rss,Dàn vũ khí trên tàu hộ vệ tàng hình Ấn Độ thăm...,https://vnexpress.net/dan-vu-khi-tren-tau-ho-v...,"Mon, 22 Jun 2026 21:13:56 +0700"
2,world,https://vnexpress.net/rss/the-gioi.rss,"Tổng Bí thư, Chủ tịch nước Tô Lâm tiếp quyền B...",https://vnexpress.net/tong-bi-thu-chu-tich-nuo...,"Mon, 22 Jun 2026 20:12:10 +0700"
3,world,https://vnexpress.net/rss/the-gioi.rss,Châu Âu oằn mình dưới nắng nóng hơn 40°C,https://vnexpress.net/chau-au-oan-minh-duoi-na...,"Mon, 22 Jun 2026 19:33:00 +0700"
4,world,https://vnexpress.net/rss/the-gioi.rss,UAV Ukraine thách thức lưới phòng không Nga,https://vnexpress.net/uav-ukraine-thach-thuc-l...,"Mon, 22 Jun 2026 19:00:00 +0700"


# Article Extraction

Mỗi tờ báo sử dụng cấu trúc HTML khác nhau.

Do đó cần xây dựng parser riêng cho từng website để trích xuất:

- Title
- Description
- Main content
- Source

Kết quả cuối cùng được chuẩn hóa thành cùng một schema.

In [6]:
def parse_soup(
    soup, 
    title_selector=".title-detail", 
    description_selector=".description", 
    paragraphs_selector=":is(.fck_detail, .item_slide_show > .desc_cation) > :is(p, h1, h2, h3, h4, h5, h6)",
    source="Chưa có thông tin"
):

    title = soup.select_one(title_selector)
    description = soup.select_one(description_selector)
    paragraphs = soup.select(paragraphs_selector)

    return {
        "title": title.text.strip() if title else "",
        "description": description.text.strip() if description else "",
        "content": re.sub(r"\s+", " ", "\n".join([
            p.get_text(" ", strip=True)
            for p in paragraphs
        ])),
        "source": source
    }

## Source-specific Scraper

Hàm này xác định nguồn báo thông qua URL và áp dụng bộ CSS selector tương ứng.

Các nguồn hiện được hỗ trợ:

- VnExpress
- Thanh Niên
- Tuổi Trẻ
- Tiền Phong
- Nhân Dân
- VTV

In [7]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "vi,en;q=0.9",
    "Referer": "https://www.google.com/",
    "Upgrade-Insecure-Requests": "1"
}

def get_article_content(url):
    res = session.get(
        url,
        headers=headers,
        timeout=60
    )

    res.encoding = "utf-8"

    soup = BeautifulSoup(res.text, "html.parser")

    if "vnexpress.net" in url:
        parsed = parse_soup(
            soup,
            title_selector=".title-detail",
            description_selector=".description",
            paragraphs_selector=":is(.fck_detail, .item_slide_show > .desc_cation) > :is(p, h1, h2, h3, h4, h5, h6)",
            source="VnExpress"
        )

    elif "thanhnien.vn" in url:
        parsed = parse_soup(
            soup,
            title_selector=".detail-title",
            description_selector=".detail-sapo",
            paragraphs_selector=".detail-content > :is(p, h1, h2, h3, h4, h5, h6)",
            source="Thanh Niên"
        )

    elif "tuoitre.vn" in url:
        parsed = parse_soup(
            soup,
            title_selector=".article-title",
            description_selector=".detail-sapo",
            paragraphs_selector=".detail-content > :is(p, h1, h2, h3, h4, h5, h6)",
            source="Tuổi trẻ"
        )

    elif "tienphong.vn" in url:
        parsed = parse_soup(
            soup,
            title_selector=".article__title",
            description_selector=".article__sapo",
            paragraphs_selector=".article__body :is(p, h1, h2, h3, h4, h5, h6)",
            source="Tiền Phong"
        )

    elif "nhandan.vn" in url:
        parsed = parse_soup(
            soup,
            title_selector=".article__title",
            description_selector=".article__sapo",
            paragraphs_selector=".article__body :is(p, h1, h2, h3, h4, h5, h6)",
            source="Nhân dân"
        )

    elif "vtv.vn" in url:
        parsed = parse_soup(
            soup,
            title_selector=".detail-title, .title",
            description_selector=".detail-sapo, .sapo",
            paragraphs_selector=":is(.detail-content, #divend) > :is(p, h1, h2, h3, h4, h5, h6)",
            source="Thời báo VTV"
        )

    else:
        return None

    parsed["url"] = url

    parsed["full_text"] = "\n".join([
        parsed["title"],
        parsed["description"],
        parsed["content"]
    ])
    if len(parsed["full_text"]) <= 300 and len(soup.text) < 1000:
        print(f"Too short {parsed["url"]}\nFull text: {parsed["full_text"]}\nSoup: {soup} \n\n")
    elif len(parsed["full_text"]) <= 300:
        print(f"Too short {parsed["url"]}\nFull text: {parsed["full_text"]}\n\n")

    parsed["summary"] = "\n".join([
        parsed["title"],
        parsed["description"],
        parsed["content"][:1000]
    ])

    return parsed

## Parallel Crawling

Việc crawl bài báo được thực hiện bằng ThreadPoolExecutor để tăng tốc quá trình tải dữ liệu.

Các bài viết:

- bị lỗi truy cập
- nội dung quá ngắn
- không parse được

sẽ bị loại bỏ khỏi tập dữ liệu.

In [8]:
def fetch_article(row):

    try:

        article = get_article_content(row["url"])

        if article and len(article["full_text"]) > 300:

            article["topic"] = row["topic"]
            article["rss_source"] = row["rss_source"]

            # ADD
            article["published"] = parser.parse(row["published"]).isoformat()

            return article

    except Exception as e:
        print("Error", row["url"], e)

        return None

In [9]:
data = []

rows = list(rss_df.to_dict("records"))

with ThreadPoolExecutor(max_workers=8) as executor:

    futures = [
        executor.submit(fetch_article, row)
        for row in rows
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):

        result = future.result()

        if result is not None:
            data.append(result)

df = pd.DataFrame(data)

print(df.shape)

  1%|          | 26/2550 [00:02<02:10, 19.38it/s]

Too short https://vnexpress.net/nhung-van-de-con-de-ngo-trong-thoa-thuan-my-iran-5088195.html
Full text: 






  9%|▉         | 225/2550 [00:26<03:16, 11.83it/s]

Too short https://nhandan.vn/special/du-lieu-tri-tue-nhan-tao/index.html
Full text: 






 13%|█▎        | 329/2550 [00:35<01:31, 24.23it/s]

...

Too short https://thanhnien.vn/xe-7-cho-gia-duoi-800-trieu-chon-mitsubishi-destinator-hay-geely-okavango-185260616172130045.htm
Full text: Xe 7 chỗ giá dưới 800 triệu, chọn Mitsubishi Destinator hay Geely Okavango?
Với hầu bao 800 triệu đồng, khách Việt có thể lựa chọn nhiều mẫu xe 7 chỗ gầm cao ở phân khúc hạng C và D, trong đó có hai mẫu xe đáng cân nhắc Mitsubishi Destinator và Geely Okavango.





 96%|█████████▌| 2445/2550 [04:33<00:08, 12.54it/s]

Too short https://thanhnien.vn/kia-sorento-hybrid-co-loi-the-nao-canh-tranh-trong-phan-khuc-xe-7-cho-185260614120957325.htm
Full text: KIA Sorento hybrid có lợi thế nào cạnh tranh trong phân khúc xe 7 chỗ?
Không chỉ bổ sung hệ truyền động hybrid, KIA Sorento 2026 còn được nâng cấp đáng kể ở thiết kế ngoại thất, khoang lái và công nghệ hỗ trợ lái, tăng sức cạnh tranh với các đối thủ trong phân khúc crossover hạng D.





100%|██████████| 2550/2550 [04:48<00:00,  8.82it/s]

(2504, 10)


# Text Embedding

Mỗi bài báo được biểu diễn thành vector ngữ nghĩa bằng mô hình:

BAAI/bge-m3

Embedding được sinh từ:

- title
- description
- phần đầu nội dung bài viết

Sau đó chuẩn hóa L2 để phục vụ tính toán cosine similarity.

In [10]:
embed_model = SentenceTransformer(
    "BAAI/bge-m3"
)

embeddings = embed_model.encode(
    df["summary"].tolist(),
    show_progress_bar=True,
    batch_size=128
)

print(embeddings.shape)

embeddings = normalize(embeddings, norm="l2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

(2504, 1024)


# Topic Clustering

Mục tiêu của bước này là nhóm các bài báo có nội dung tương đồng thành các cụm chủ đề.

Quy trình:

1. Giảm chiều embedding bằng UMAP.
2. Chạy HDBSCAN trên không gian giảm chiều.
3. Quét nhiều cấu hình HDBSCAN.
4. Tự động chọn cấu hình tốt nhất.

Tiêu chí lựa chọn:

- Cluster coverage cao
- Noise ratio thấp
- Silhouette score tốt
- Tránh cụm quá lớn
- Tránh quá ít hoặc quá nhiều cụm

Việc clustering được thực hiện trên không gian UMAP nhưng đánh giá độ nhất quán ngữ nghĩa sử dụng embedding gốc.

## UMAP Dimensionality Reduction

UMAP được sử dụng để:

- giảm nhiễu trong không gian embedding
- giữ cấu trúc lân cận của dữ liệu
- cải thiện hiệu quả của HDBSCAN

Không gian 30 chiều được sử dụng cho clustering thay vì embedding gốc.

In [11]:
# =========================================================
# OPTIMIZED CLUSTERING: UMAP + HDBSCAN AUTO-SWEEP
# =========================================================
# Mục tiêu:
# - Giảm noise_ratio so với cấu hình cũ.
# - Không ép toàn bộ bài báo vào 1 cụm khổng lồ.
# - Ưu tiên cụm có độ phủ tốt nhưng vẫn giữ tính tách biệt chủ đề.
#
# Lưu ý:
# - embeddings vẫn là vector BAAI/bge-m3 đã normalize L2.
# - UMAP chỉ dùng để tạo không gian clustering ổn định hơn.
# - Khi chọn bài đại diện cho LLM, vẫn dùng embeddings gốc để giữ ngữ nghĩa.

RANDOM_STATE = 42

reducer = umap.UMAP(
    n_neighbors=15,
    n_components=30,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_STATE
)

embedding_2d = reducer.fit_transform(embeddings)
df["umap_x"] = embedding_2d[:, 0]
df["umap_y"] = embedding_2d[:, 1]

USE_UMAP_FOR_CLUSTERING = True

if USE_UMAP_FOR_CLUSTERING:
    cluster_input = embedding_2d
    print("UMAP clustering input:", cluster_input.shape)

else:
    cluster_input = embeddings
    print("Raw embedding clustering input:", cluster_input.shape)

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP clustering input: (2504, 30)


In [12]:
def summarize_labels(candidate_labels):
    unique_clusters = [x for x in np.unique(candidate_labels) if x != -1]

    cluster_sizes = [
        int(np.sum(candidate_labels == cid))
        for cid in unique_clusters
    ]

    n_clusters = len(unique_clusters)
    noise_articles = int(np.sum(candidate_labels == -1))
    noise_ratio = float(np.mean(candidate_labels == -1))
    clustered_articles = int(np.sum(candidate_labels != -1))
    largest_cluster_ratio = (
        max(cluster_sizes) / len(candidate_labels)
        if cluster_sizes
        else 0
    )

    return {
        "n_clusters": n_clusters,
        "clustered_articles": clustered_articles,
        "noise_articles": noise_articles,
        "noise_ratio": noise_ratio,
        "largest_cluster_ratio": largest_cluster_ratio,
        "cluster_sizes": cluster_sizes
    }


def safe_silhouette(X, candidate_labels):
    mask = candidate_labels != -1
    valid_labels = candidate_labels[mask]

    if len(set(valid_labels)) < 2 or len(valid_labels) < 3:
        return None

    try:
        return float(
            silhouette_score(
                X[mask],
                valid_labels,
                metric="euclidean"
            )
        )
    except Exception:
        return None

def cluster_coherence(cluster_embs):
    centroid = cluster_embs.mean(axis=0)

    sims = cosine_similarity(
        cluster_embs,
        [centroid]
    )

    return sims.mean()


def weighted_cluster_coherence(labels):
    cluster_scores = []
    cluster_coherence_map = {}
    
    for cid in np.unique(labels):
    
        if cid == -1:
            continue
    
        # Coherence dùng embeddings gốc để đo tính nhất quán ngữ nghĩa
        cluster_embs = embeddings[labels == cid]
    
        if len(cluster_embs) < 2:
            cluster_coherence_map[cid] = 1.0
    
        cluster_coherence_map[cid] = score = cluster_coherence(cluster_embs)
    
        cluster_scores.append({
            "cluster": int(cid),
            "size": int(len(cluster_embs)),
            "coherence": float(score)
        })
    
    if cluster_scores:
        weighted_mean = np.average(
            [x["coherence"] for x in cluster_scores],
            weights=[x["size"] for x in cluster_scores]
        )
    else:
        weighted_mean = None
    
    return weighted_mean


## HDBSCAN Configuration Search

HDBSCAN rất nhạy với các tham số:

- min_cluster_size
- min_samples
- cluster_selection_method
- cluster_selection_epsilon

Thay vì cố định một cấu hình, notebook thử nhiều cấu hình khác nhau và chọn cấu hình có objective score cao nhất.

In [13]:
candidate_configs = []

for size in range(3, 20):
    for samples in [1,2]:
        candidate_configs.append({
            "min_cluster_size": size,
            "min_samples": samples,
            "cluster_selection_method": "eom",
            "cluster_selection_epsilon": 0.0
        })

### Objective Function

Objective được thiết kế để cân bằng:

- Coverage cao
- Noise thấp
- Cluster quality tốt

Đồng thời áp dụng penalty cho:

- cụm khổng lồ
- quá ít cụm
- quá nhiều cụm

In [14]:
candidate_results = []

for cfg in candidate_configs:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=cfg["min_cluster_size"],
        min_samples=cfg["min_samples"],
        metric="euclidean",
        cluster_selection_method=cfg["cluster_selection_method"],
        cluster_selection_epsilon=cfg["cluster_selection_epsilon"]
    )

    candidate_labels = clusterer.fit_predict(cluster_input)
    summary = summarize_labels(candidate_labels)
    sil = safe_silhouette(cluster_input, candidate_labels)
    coherence = weighted_cluster_coherence(candidate_labels)
    
    # Score chọn cấu hình:
    # - ưu tiên coverage cao / noise thấp
    # - cộng điểm nếu silhouette tốt
    # - phạt nếu tạo cụm khổng lồ hoặc quá ít/quá nhiều cụm
    coverage = 1 - summary["noise_ratio"]
    sil_score = max(sil, 0) if sil is not None else 0
    coherence_score = coherence if coherence is not None else 0

    objective = 0.35 * coverage + 0.20 * sil_score + 0.45 * coherence_score

    if summary["noise_ratio"] > 0.2 or coherence_score < 0.7:
        continue
    if summary["n_clusters"] < 60 or summary["n_clusters"] > 180:
        continue

    # if summary["n_clusters"] < 8:
    #     objective -= 0.20

    # if summary["n_clusters"] > 80:
    #     objective -= 0.002 * (summary["n_clusters"] - 80)

    # if summary["largest_cluster_ratio"] > 0.35:
    #     objective -= 0.60 * (summary["largest_cluster_ratio"] - 0.35)

    result = {
        **cfg,
        **summary,
        "silhouette": sil,
        "coherence": coherence,
        "objective": objective,
        "labels": candidate_labels,
        "clusterer": clusterer
    }

    candidate_results.append(result)

candidate_results = sorted(
    candidate_results,
    key=lambda x: x["objective"],
    reverse=True
)

print("=== HDBSCAN CONFIG SEARCH RESULTS ===")
for i, r in enumerate(candidate_results, 1):
    print(
        i,
        "| size=", r["min_cluster_size"],
        "| samples=", r["min_samples"],
        "| method=", r["cluster_selection_method"],
        "| eps=", f'{r["cluster_selection_epsilon"]:.2}',
        "| clusters=", f'{r["n_clusters"]:3}',
        "| noise=", f'{r["noise_ratio"]:.2%}',
        "| largest=", f'{r["largest_cluster_ratio"]:.2%}',
        "| silhouette=", None if r["silhouette"] is None else f'{r["silhouette"]:.4}',
        "| coherence=", None if r["coherence"] is None else f'{r["coherence"]:.4}',
        "| objective=", f'{r["objective"]:.4}',
    )

best_result = candidate_results[0]
df["cluster"] = labels = best_result["labels"]

print("\n=== SELECTED CONFIG ===")
print({
    "min_cluster_size": best_result["min_cluster_size"],
    "min_samples": best_result["min_samples"],
    "cluster_selection_method": best_result["cluster_selection_method"],
    "cluster_selection_epsilon": f'{best_result["cluster_selection_epsilon"]:.2}',
    "num_clusters": best_result["n_clusters"],
    "noise_ratio": f'{best_result["noise_ratio"]:.2%}',
    "clustered_articles": best_result["clustered_articles"],
    "noise_articles": best_result["noise_articles"],
    "largest_cluster_ratio": f'{best_result["largest_cluster_ratio"]:.2%}',
    "silhouette": f'{best_result["silhouette"]:.4}',
    "coherence": f'{best_result["coherence"]:.4}',
})


=== HDBSCAN CONFIG SEARCH RESULTS ===
1 | size= 5 | samples= 2 | method= eom | eps= 0.0 | clusters= 169 | noise= 16.29% | largest= 3.55% | silhouette= 0.5654 | coherence= 0.7799 | objective= 0.757
2 | size= 6 | samples= 1 | method= eom | eps= 0.0 | clusters= 144 | noise= 15.69% | largest= 3.59% | silhouette= 0.5444 | coherence= 0.7686 | objective= 0.7498
3 | size= 8 | samples= 1 | method= eom | eps= 0.0 | clusters= 119 | noise= 16.45% | largest= 2.56% | silhouette= 0.5766 | coherence= 0.7582 | objective= 0.7489
4 | size= 9 | samples= 2 | method= eom | eps= 0.0 | clusters=  98 | noise= 16.85% | largest= 3.55% | silhouette= 0.6043 | coherence= 0.7487 | objective= 0.7488
5 | size= 13 | samples= 1 | method= eom | eps= 0.0 | clusters=  71 | noise= 10.86% | largest= 5.51% | silhouette= 0.5515 | coherence= 0.725 | objective= 0.7485
6 | size= 8 | samples= 2 | method= eom | eps= 0.0 | clusters= 114 | noise= 17.33% | largest= 3.55% | silhouette= 0.5941 | coherence= 0.7563 | objective= 0.7485
7 |

# Clustering Evaluation

Sau khi chọn cấu hình tốt nhất, nhiều metric được tính toán để đánh giá chất lượng phân cụm:

- Number of clusters
- Cluster coverage
- Noise ratio
- Silhouette score
- DBCV
- Davies-Bouldin Index

Các metric này sẽ được export vào file kết quả cuối cùng.

In [15]:
mask = labels != -1
valid_labels = labels[mask]

num_clusters = int(len([x for x in np.unique(labels) if x != -1]))
noise_articles = int(np.sum(labels == -1))
clustered_articles = int(np.sum(labels != -1))
noise_ratio = float(np.mean(labels == -1))
cluster_coverage = float(1 - noise_ratio)

print(f"Number of clusters: {num_clusters}")
print(f"Clustered articles: {clustered_articles}")
print(f"Noise articles: {noise_articles}")
print(f"Noise ratio: {noise_ratio:.2%}")
print(f"Cluster coverage: {cluster_coverage:.2%}")

# Dùng cluster_input để đánh giá vì HDBSCAN chạy trên không gian này.
if len(set(valid_labels)) >= 2 and len(valid_labels) >= 3:
    silhouette = silhouette_score(
        cluster_input[mask],
        valid_labels,
        metric="euclidean"
    )
else:
    silhouette = None

print(f"Silhouette score: {silhouette:.4}")

try:
    dbcv_score = hdbscan.validity_index(
        cluster_input[mask].astype(np.float64),
        valid_labels,
        metric="euclidean"
    )
except Exception:
    dbcv_score = None

print(f"DBCV: {dbcv_score:.4}")

try:
    db_index = davies_bouldin_score(
        cluster_input[mask],
        valid_labels
    )
except Exception:
    db_index = None

print(f"Davies-Bouldin Index: {db_index:.4}")

Number of clusters: 169
Clustered articles: 2096
Noise articles: 408
Noise ratio: 16.29%
Cluster coverage: 83.71%
Silhouette score: 0.5654
DBCV: 0.5284
Davies-Bouldin Index: 0.552


## Semantic Coherence

Độ nhất quán ngữ nghĩa của cụm được đo bằng:

Cosine Similarity giữa:

- embedding từng bài báo
- centroid của cụm

Giá trị càng cao cho thấy các bài trong cụm càng nói về cùng một chủ đề.

In [16]:
cluster_scores = []
cluster_coherence_map = {}

for cid in np.unique(labels):

    if cid == -1:
        continue

    # Coherence dùng embeddings gốc để đo tính nhất quán ngữ nghĩa
    cluster_embs = embeddings[labels == cid]

    if len(cluster_embs) < 2:
        cluster_coherence_map[cid] = 1.0

    cluster_coherence_map[cid] = score = cluster_coherence(cluster_embs)

    cluster_scores.append({
        "cluster": int(cid),
        "size": int(len(cluster_embs)),
        "coherence": float(score)
    })

cluster_scores = sorted(
    cluster_scores,
    key=lambda x: x["coherence"],
    reverse=True
)

print("\nTop coherence clusters:")
for row in cluster_scores[:10]:
    print(row)

if cluster_scores:
    weighted_mean = np.average(
        [x["coherence"] for x in cluster_scores],
        weights=[x["size"] for x in cluster_scores]
    )
else:
    weighted_mean = None

print(f"Weighted mean coherence: {weighted_mean:.4}")



Top coherence clusters:
{'cluster': 9, 'size': 15, 'coherence': 0.9176565408706665}
{'cluster': 145, 'size': 9, 'coherence': 0.9174995422363281}
{'cluster': 32, 'size': 5, 'coherence': 0.9123613238334656}
{'cluster': 18, 'size': 7, 'coherence': 0.9119011759757996}
{'cluster': 17, 'size': 7, 'coherence': 0.910605788230896}
{'cluster': 4, 'size': 8, 'coherence': 0.9071201086044312}
{'cluster': 26, 'size': 9, 'coherence': 0.9034498929977417}
{'cluster': 33, 'size': 5, 'coherence': 0.8955832719802856}
{'cluster': 34, 'size': 10, 'coherence': 0.8949955105781555}
{'cluster': 10, 'size': 5, 'coherence': 0.8917152285575867}
Weighted mean coherence: 0.7799


In [17]:
for cluster_id in sorted(df["cluster"].unique())[:20]:

    print("=" * 80)
    print("CLUSTER", cluster_id)

    subset = df[df["cluster"] == cluster_id]

    for title in subset["title"].head(5):
        print("-", title)

CLUSTER -1
- Giáo sư 'chip ung thư' rời Đại học New York về Trung Quốc
- Dàn vũ khí trên tàu hộ vệ tàng hình Ấn Độ thăm TP HCM
- Những người Mỹ hoang mang vì bị Canada đòi lại quốc tịch
- Người bố trăm cân giảm 27 kg để bơi đua với con gái
- Lương 104.000 USD một năm vẫn là 'thu nhập thấp' ở quận Cam
CLUSTER 0
- Thủ tướng Anh nghẹn ngào tuyên bố từ chức
- Người Ấn biểu tình, đòi Bộ trưởng Giáo dục từ chức sau bê bối lộ đề thi
- Nóng: Thủ tướng Anh thông báo từ chức
- Rộ tin Thủ tướng Anh Keir Starmer sắp từ chức
- Thủ tướng Anh từ chức, xúc động cảm ơn vợ
CLUSTER 1
- Dịch cúm bùng phát tại căn cứ Mỹ sau khi bỏ quy định tiêm vaccine
- Mỹ xem xét vaccine cúm đầu tiên sử dụng công nghệ mRNA
- Australia xác nhận trường hợp cúm gia cầm H5N1 đầu tiên
- Bùng phát cúm trong 'thành phố quân sự' của Mỹ, 150 tân binh mắc bệnh
- Australia xác nhận trường hợp cúm gia cầm H5N1 đầu tiên
CLUSTER 2
- Hơn 22.300 thí sinh đổ về kỳ thi vào 8 trường công an
- Gần 23.000 thí sinh dồn lực vào kỳ thi của Bộ Q

# Trend Generation Model

Sau khi clustering, mỗi cụm sẽ được tóm tắt thành một xu hướng (trend).

Mô hình sử dụng:

Qwen2.5-7B-Instruct

Được load dưới dạng 4-bit quantization nhằm giảm VRAM và tăng tốc inference.

In [18]:
model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

# Trend Summarization

Đối với mỗi cụm:

1. Chọn các bài đại diện.
2. Xây dựng prompt.
3. Gửi tới LLM.
4. Sinh một câu mô tả xu hướng.

Mỗi trend được biểu diễn dưới dạng một câu tiếng Việt ngắn gọn.

## Representative Article Selection

Không phải toàn bộ bài viết trong cụm đều được đưa vào LLM.

Mỗi cụm sẽ chọn một tập bài đại diện dựa trên:

- Độ gần centroid
- Diversity constraint

Điều này giúp:

- giảm token
- tránh bài viết trùng lặp
- giữ được các khía cạnh quan trọng của chủ đề

In [19]:
# =========================================================
# CONFIG
# =========================================================

TOP_K = 5
DIVERSITY_THRESHOLD = 0.9

# =========================================================
# HELPER
# =========================================================

def normalize(v):
    norm = np.linalg.norm(v)

    if norm == 0:
        return v

    return v / norm


def select_representative_docs(
    cluster_embeddings,
    cluster_indices,
    top_k=5,
    diversity_threshold=0.9
):
    """
    Chọn top-k bài:
    - gần centroid
    - có diversity nhẹ
    """

    centroid = np.mean(cluster_embeddings, axis=0)
    centroid = normalize(centroid)

    normalized_embs = np.array([
        normalize(x)
        for x in cluster_embeddings
    ])

    sims = cosine_similarity(
        [centroid],
        normalized_embs
    )[0]

    sorted_idx = np.argsort(-sims)

    selected_local_idx = []

    for idx in sorted_idx:

        current_emb = normalized_embs[idx]

        too_similar = False

        for selected_idx in selected_local_idx:

            sim = cosine_similarity(
                [current_emb],
                [normalized_embs[selected_idx]]
            )[0][0]

            if sim >= diversity_threshold:
                too_similar = True
                break

        if not too_similar:
            selected_local_idx.append(idx)

        if len(selected_local_idx) >= top_k:
            break

    selected_global_indices = [
        cluster_indices[i]
        for i in selected_local_idx
    ]

    return selected_global_indices

## Trend Generation Pipeline

Mỗi cụm chủ đề được chuyển thành một prompt gồm các bài báo đại diện.

Quy trình sinh trend:

1. Xây dựng prompt từ tiêu đề và mô tả bài báo.
2. Áp dụng chat template của mô hình.
3. Sinh kết quả theo batch để tăng throughput.
4. Trích xuất phần văn bản được sinh ra.
5. Thu được một câu mô tả xu hướng cho mỗi cụm.

System prompt được thiết kế để buộc mô hình trả về đúng một câu tóm tắt ngắn gọn, phù hợp cho dashboard và trực quan hóa xu hướng.

In [20]:
def generate_batch(prompts, batch_size=4):

    results = []

    system_prompt = (
        "Bạn là AI chuyên tổng hợp xu hướng báo chí.\n"
        "Chỉ trả về đúng 1 câu summary.\n"
        "Không giải thích.\n"
        "Không bullet point.\n"
        "Không nhắc lại yêu cầu."
    )

    for i in tqdm(range(0, len(prompts), batch_size)):

        batch_prompts = prompts[i:i + batch_size]

        # ==========================================
        # BUILD CHAT MESSAGES
        # ==========================================

        message_batch = []

        for prompt in batch_prompts:

            messages = [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]

            message_batch.append(messages)

        # ==========================================
        # APPLY CHAT TEMPLATE
        # ==========================================

        text_batch = tokenizer.apply_chat_template(
            message_batch,
            tokenize=False,
            add_generation_prompt=True
        )

        # ==========================================
        # TOKENIZE
        # ==========================================

        inputs = tokenizer(
            text_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(model.device)

        # ==========================================
        # GENERATE
        # ==========================================

        with torch.inference_mode():

            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True
            )

        # ==========================================
        # REMOVE PROMPT TOKENS
        # ==========================================

        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids
            in zip(inputs.input_ids, outputs)
        ]

        # ==========================================
        # DECODE
        # ==========================================

        decoded = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        results.extend([
            d.strip()
            for d in decoded
        ])

        torch.cuda.empty_cache()

    return results

def build_trend_prompt(representative_articles):

    prompt = """Tóm tắt các bài báo sau thành đúng 1 câu tiếng Việt ngắn.

Không bullet point.
Không giải thích.
Không nhắc lại yêu cầu.

Các bài báo:
"""

    for i, article in enumerate(representative_articles, 1):

        prompt += f"""
{i}.
Tiêu đề: {article["title"]}

Tóm tắt:
{article["description"]}
"""

    prompt += "\nKết quả:\n"
    return prompt

In [21]:
# =========================================================
# BUILD CLUSTER DATA
# =========================================================

cluster_data = {}

cluster_docs = defaultdict(list)

for idx, row in df.iterrows():
    cluster_docs[row["cluster"]].append(idx)

for cluster_id, indices in cluster_docs.items():
    # skip noise
    if cluster_id == -1:
        continue

    subset = df.iloc[indices]

    cluster_embs = embeddings[indices]

    # =====================================================
    # SELECT REPRESENTATIVE ARTICLES
    # =====================================================

    representative_indices = select_representative_docs(
        cluster_embs,
        indices,
        top_k=TOP_K,
        diversity_threshold=DIVERSITY_THRESHOLD
    )

    all_articles = []
    representative_articles = []
    
    for idx in indices:
        row = df.iloc[idx]

        data_to_save = {
            "index": int(idx),
            "title": row["title"],
            "description": row["description"],
            "url": row["url"],
            "topic": row["topic"],
            "published": row.get("published", ""),
            "source": row.get("source", row.get("rss_source", "")),
            "umap_x": float(row["umap_x"]),
            "umap_y": float(row["umap_y"]),
        }

        all_articles.append(data_to_save)
        if idx in representative_indices:
            representative_articles.append(data_to_save)

    # =====================================================
    # BUILD PROMPT
    # =====================================================

    prompt = build_trend_prompt(
        representative_articles
    )

    # =====================================================
    # SAVE CLUSTER
    # =====================================================

    cluster_data[cluster_id] = {
        "cluster_id": int(cluster_id),
        "num_articles": len(indices),
        "article_indices": indices,
        "coherence": float(cluster_coherence_map[cluster_id]),
        "representative_indices": representative_indices,
        "representative_articles": representative_articles,
        "all_articles": all_articles,
        "prompt": prompt,
        "trend": None
    }

# =========================================================
# GENERATE TRENDS
# =========================================================

cluster_ids = list(cluster_data.keys())

prompts = [
    cluster_data[cid]["prompt"]
    for cid in cluster_ids
]

trends = generate_batch(
    prompts,
    batch_size=8
)

for cid, trend in zip(cluster_ids, trends):
    cluster_data[cid]["trend"] = trend

# =========================================================
# SORT
# =========================================================

sorted_clusters = sorted(
    cluster_data.items(),
    key=lambda x: x[1]["num_articles"],
    reverse=True
)

cluster_data = {
    i: data
    for i, (_, data) in enumerate(sorted_clusters)
}

# =========================================================
# PREVIEW
# =========================================================

for cid, data in list(cluster_data.items())[:20]:
    print("=" * 80)
    print("CLUSTER:", cid)

    print("TREND:")
    print(data["trend"])

    print(f"\nNUMBER OF ARTICLES: {data['num_articles']}")

    print("\nREPRESENTATIVE ARTICLES:")
    for article in data["representative_articles"]:

        print("-", article["title"])


  0%|          | 0/22 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

100%|██████████| 22/22 [05:56<00:00, 16.21s/it]

CLUSTER: 0
TREND:
Thị trường ô tô Việt Nam sôi động với nhiều mẫu SUV mới từ phân khúc phổ thông đến hạng sang như Audi Q5, Mazda CX-60 và Lynk & Co 900.

NUMBER OF ARTICLES: 89

REPRESENTATIVE ARTICLES:
- Audi Q5 thế hệ mới giá từ 2,8 tỷ đồng
- Mazda CX-60 giá 1,699 tỷ đồng tại Việt Nam
- Lynk & Co 900 - thách thức xe sang bằng trang bị
- Mazda CX-60 chốt giá 1,699 tỉ ở Việt Nam: Máy hybrid 280 mã lực, AWD, to cỡ Mercedes-Benz GLC
- Loạt SUV mới cập bến Việt Nam
CLUSTER: 1
TREND:
Mỹ và Iran bắt đầu đàm phán tại Thụy Sĩ với sự tham gia của bên trung gian, đạt tiến triển nhưng vẫn gặp bế tắc do lời đe dọa của Tổng thống Trump.

NUMBER OF ARTICLES: 56

REPRESENTATIVE ARTICLES:
- Mỹ, Iran bắt đầu đàm phán ở Thụy Sĩ
- Mỹ và Iran bắt đầu đàm phán
- Ông Vance khen đàm phán 'đạt tiến triển lớn', nhưng eo biển Hormuz vẫn đóng
- Đàm phán Mỹ - Iran tại Thụy Sĩ sẽ thảo luận những vấn đề gì?
- Đàm phán Mỹ - Iran đình trệ sau lời đe dọa của Tổng thống Trump
CLUSTER: 2
TREND:
Chuyển đổi số trở thành

# Export Results

Kết quả cuối cùng được lưu vào:

trends.json

Bao gồm:

- Metadata dự án
- Cấu hình mô hình
- Clustering metrics
- Danh sách trends
- Danh sách bài báo thuộc từng trend

File này có thể được sử dụng trực tiếp cho:

- Dashboard
- Search API
- Trend monitoring system
- Data visualization

In [22]:
export_trends = []

for rank, data in cluster_data.items():
    export_trends.append({
        "rank": int(rank) + 1,
        "cluster_id": int(data["cluster_id"]),
        "coherence": data["coherence"],
        "trend_name": data["trend"],
        "article_count": int(data["num_articles"]),
        "representative_articles": data["representative_articles"],
        "articles": data["all_articles"]
    })

payload = {
    "project_name": "News Trend Detection",
    "generated_at": datetime.now().isoformat(),
    "model": {
        "embedding_model": "BAAI/bge-m3",
        "llm_model": "Qwen/Qwen2.5-7B-Instruct",
        "quantization": "4-bit"
    },
    "clustering_config": {
        "use_umap_for_clustering": bool(USE_UMAP_FOR_CLUSTERING),
        "selected_hdbscan": {
            "min_cluster_size": int(best_result["min_cluster_size"]),
            "min_samples": int(best_result["min_samples"]),
            "cluster_selection_method": best_result["cluster_selection_method"],
            "cluster_selection_epsilon": float(best_result["cluster_selection_epsilon"])
        }
    },
    "metrics": {
        "total_rss_articles": int(len(rss_df)),
        "total_scraped_articles": int(len(df)),
        "num_clusters": int(num_clusters),
        "clustered_articles": int(clustered_articles),
        "noise_articles": int(noise_articles),
        "noise_ratio": float(noise_ratio),
        "cluster_coverage": float(cluster_coverage),
        "weighted_coherence": None if weighted_mean is None else float(weighted_mean),
        "silhouette_score": None if silhouette is None else float(silhouette),
        "dbcv": None if dbcv_score is None else float(dbcv_score),
        "davies_bouldin_index": None if db_index is None else float(db_index)
    },
    "trends": export_trends
}

with open("trends.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("Exported trends.json")
print("Metrics:")
print(json.dumps(payload["metrics"], ensure_ascii=False, indent=4))


Exported trends.json
Metrics:
{
    "total_rss_articles": 2550,
    "total_scraped_articles": 2504,
    "num_clusters": 169,
    "clustered_articles": 2096,
    "noise_articles": 408,
    "noise_ratio": 0.16293929712460065,
    "cluster_coverage": 0.8370607028753994,
    "weighted_coherence": 0.7798864853120487,
    "silhouette_score": 0.565422534942627,
    "dbcv": 0.5283942569566685,
    "davies_bouldin_index": 0.5520380507516508
}
